In [ ]:
# ─── 1. Install ───────────────────────────────────────────────────────────────
!pip uninstall -y peft accelerate transformers -q
!pip install -q \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  safetensors==0.4.2 \
  scikit-learn \
  Pillow \
  numpy==1.26.4

import os
os.kill(os.getpid(), 9)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.


In [ ]:
# ─── 2. Setup ─────────────────────────────────────────────────────────────────
import os, time, functools
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_ALLOC_CONF"]   = "expandable_segments:True"

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/CLIP_ViT_B32_Prompt_IFND"
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── 3. Imports ───────────────────────────────────────────────────────────────
import torch
import numpy as np
from PIL import Image
from torch import nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import (CLIPProcessor, CLIPModel, TrainingArguments,
                          Trainer, EarlyStoppingCallback)
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.trainer_utils import get_last_checkpoint
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ─── 4. Load dataset ──────────────────────────────────────────────────────────
print("⏳ Loading dataset...")
hf_dataset = load_dataset("Nhat243/IFND-multimodal")
print(hf_dataset)

# ─── 5. Preprocess ────────────────────────────────────────────────────────────
model_name = "openai/clip-vit-base-patch32"
processor  = CLIPProcessor.from_pretrained(model_name)

def preprocess(example):
    try:
        image = example['image'].convert("RGB")
    except:
        image = Image.new("RGB", (224, 224), (128, 128, 128))
    inputs = processor(
        text=str(example['text']),
        images=image,
        truncation=True,
        padding="max_length",
        max_length=77,
        return_tensors="pt"
    )
    return {
        "input_ids":      inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "pixel_values":   inputs["pixel_values"][0],
        "labels":         int(example["label"])
    }

print("⏳ Preprocessing dataset...")
encoded_dataset = hf_dataset.map(
    preprocess,
    remove_columns=hf_dataset["train"].column_names,
    desc="Preprocessing"
)
encoded_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "pixel_values", "labels"]
)

train_dataset = encoded_dataset["train"]
val_dataset   = encoded_dataset["validation"]
test_dataset  = encoded_dataset["test"]
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ─── 6. Model ─────────────────────────────────────────────────────────────────
PROMPT_LENGTH = 5

class CLIPWithPrompt(nn.Module):
    def __init__(self, model_name, num_labels=2, prompt_length=5):
        super().__init__()
        self.clip          = CLIPModel.from_pretrained(model_name)
        self.prompt_length = prompt_length

        # Freeze toàn bộ CLIP
        for param in self.clip.parameters():
            param.requires_grad = False

        self.hidden_dim  = self.clip.config.text_config.hidden_size
        self.max_pos     = self.clip.text_model.embeddings.position_embedding.num_embeddings  # 77
        self.soft_prompt = nn.Parameter(torch.randn(prompt_length, self.hidden_dim))
        self.classifier  = nn.Linear(self.clip.config.projection_dim * 2, num_labels)

    def forward(self, input_ids=None, attention_mask=None,
                pixel_values=None, labels=None):
        batch_size = input_ids.size(0)
        device     = input_ids.device

        # Token embeddings
        token_embeddings = self.clip.text_model.embeddings.token_embedding(input_ids)

        # Prepend soft prompts
        soft_prompt   = self.soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)
        hidden_states = torch.cat([soft_prompt, token_embeddings], dim=1)

        # Truncate đến max_pos để không vượt quá position embedding
        hidden_states  = hidden_states[:, :self.max_pos, :]
        attention_mask = torch.cat([
            torch.ones(batch_size, self.prompt_length, device=device),
            attention_mask
        ], dim=1)[:, :self.max_pos]

        seq_length = hidden_states.size(1)  # = 77

        # Position embeddings
        position_ids  = torch.arange(seq_length, device=device).unsqueeze(0)
        hidden_states = hidden_states + \
            self.clip.text_model.embeddings.position_embedding(position_ids)

        # Causal mask (batch_size, 1, seq_len, seq_len)
        causal_mask = torch.triu(
            torch.full((seq_length, seq_length), float("-inf"), device=device), diagonal=1
        ).unsqueeze(0).unsqueeze(0).expand(batch_size, 1, seq_length, seq_length)

        # Attention mask (batch_size, 1, seq_len, seq_len)
        attention_mask = attention_mask[:, None, None, :] \
            .expand(batch_size, 1, seq_length, seq_length)

        encoder_outputs = self.clip.text_model.encoder(
            inputs_embeds=hidden_states,
            attention_mask=attention_mask,
            causal_attention_mask=causal_mask,
        )

        hidden_states = self.clip.text_model.final_layer_norm(encoder_outputs[0])

        # EOS position — clamp để tránh out of bounds
        eos_token_id  = self.clip.config.text_config.eos_token_id
        eos_positions = (input_ids == eos_token_id).float().argmax(dim=1) + self.prompt_length
        eos_positions = eos_positions.clamp(max=seq_length - 1)
        pooled_output = hidden_states[torch.arange(batch_size, device=device), eos_positions]
        text_features = self.clip.text_projection(pooled_output)

        # Image features
        image_outputs  = self.clip.vision_model(pixel_values=pixel_values)
        image_features = self.clip.visual_projection(image_outputs.pooler_output)

        # Fusion + classify
        fused  = torch.cat([text_features, image_features], dim=1)
        logits = self.classifier(fused)
        loss   = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits)

torch.cuda.empty_cache()
import gc; gc.collect()

model = CLIPWithPrompt(model_name, prompt_length=PROMPT_LENGTH).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.4f}%)")

# ─── 7. Metrics ───────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds  = np.argmax(probs, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": p,
        "recall":    r,
        "f1":        f1,
        "auc":       roc_auc_score(labels, probs[:, 1])
    }

# ─── 8. Training Args ─────────────────────────────────────────────────────────
torch.load = functools.partial(torch.load, weights_only=False)

training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none"
)

# ─── 9. Trainer ───────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

last_checkpoint = get_last_checkpoint(SAVE_DIR)
if last_checkpoint:
    print(f"▶️ Resuming from: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("🆕 Training from scratch")
    trainer.train()

trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("✅ Saved to:", SAVE_DIR)

# ─── 10. Latency + VRAM ───────────────────────────────────────────────────────
model.eval()
dummy_image = Image.new("RGB", (224, 224), (128, 128, 128))
dummy_text  = "This is a sample news headline for inference measurement."
inputs = processor(
    text=dummy_text, images=dummy_image,
    truncation=True, padding="max_length",
    max_length=77, return_tensors="pt"
)
input_ids      = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)
pixel_values   = inputs["pixel_values"].to(device)

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

with torch.no_grad():
    for _ in range(10):
        model(input_ids=input_ids, attention_mask=attention_mask,
              pixel_values=pixel_values)

latencies = []
with torch.no_grad():
    for _ in range(100):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(input_ids=input_ids, attention_mask=attention_mask,
              pixel_values=pixel_values)
        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
vram = torch.cuda.max_memory_allocated() / 1024**2 if device == "cuda" else 0

print(f"\n{'='*40}")
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Latency median  : {np.median(latencies):.2f} ms")
print(f"VRAM (peak)     : {vram:.1f} MB")
print(f"{'='*40}")

# ─── 11. Test Evaluation ──────────────────────────────────────────────────────
print("\n📊 Evaluating on test set...")
test_results = trainer.predict(test_dataset)
logits = test_results.predictions
labels = test_results.label_ids
probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
preds  = np.argmax(probs, axis=1)

p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
print(f"\n{'='*40}")
print(f"Test Accuracy : {accuracy_score(labels, preds)*100:.2f}%")
print(f"Test Precision: {p*100:.2f}%")
print(f"Test Recall   : {r*100:.2f}%")
print(f"Test F1       : {f1*100:.2f}%")
print(f"Test AUC      : {roc_auc_score(labels, probs[:, 1]):.5f}")
print(f"{'='*40}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda
⏳ Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 8416
    })
    validation: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 1052
    })
    test: Dataset({
        features: ['id', 'text', 'image', 'label'],
        num_rows: 1053
    })
})


...

⏳ Preprocessing dataset...
Train: 8416 | Val: 1052 | Test: 1053
Total params    : 151,281,923
Trainable params: 4,610 (0.0030%)
🆕 Training from scratch


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.575300,0.535013,0.766160,0.766160,1.000000,0.867600,0.583230
2,0.527000,0.517951,0.766160,0.766160,1.000000,0.867600,0.644677
3,0.516600,0.507301,0.766160,0.766160,1.000000,0.867600,0.680297


✅ Saved to: /content/drive/MyDrive/CLIP_ViT_B32_Prompt_IFND

Total params    : 151,281,923
Trainable params: 4,610
Latency median  : 45.88 ms
VRAM (peak)     : 605.2 MB

📊 Evaluating on test set...



Test Accuracy : 76.64%
Test Precision: 76.64%
Test Recall   : 100.00%
Test F1       : 86.77%
Test AUC      : 0.58375
